In [ ]:
class OrcaCalculation:
    """
    """

    def __init__(self, id):
        self.id = id
        self.opt = True
        self.optTheory = "B3LYP D4 def2-svp"
        self.optExtraBasis = {"Ni" : "def2-TZVP"}
        self.freq = False
        self.propTheory = "wb97m-D4 def2-tzvp"
        self.nmrTheory = self.propTheory
        self.calcNBO = True
        self.calcHirshfeld = True
        self.calcChelpg = True
        self.calcNMR = True
        self.calcPolarizatability = True
        self.calcVerticalRedox = False
        self.nproc = 8
        self.mem = 1125  # in MB
        self.charge = 0
        self.multiplicity = 1
        self.geomConstraints = None
        self.confSearch = False
        self.goat = "GOAT"
        self.confSearchMethod = "XTB"
        self.solvent = None  # e.g., "water", "methanol", "acetonitrile", etc.

    def set_coords_xyzblock(self, xyzblock):
        self.xyzblock = xyzblock
        lines = xyzblock.strip().splitlines()

        coords = ""
        for line in lines:
            coord = line.split()
            if len(coord) != 4:
                continue
            atom, x, y, z = coord
            coords += f" {atom}   {x}   {y}   {z}\n"
        self.coords = coords

    def set_coords_file(self, filename):
        with open(filename, "r") as f:
            xyzblock = f.read()
        self.set_coords_xyzblock(xyzblock)

    def set_coords_mol(self, mol):
        self.set_coords_xyzblock(mol.to_xyz_block())


    def write_input_files(self):
        # conf search input file
        inputfiles = []

        if self.confSearch:
            inputfile = f"! &{self.confSearchMethod} &{self.confSearchMethod}\n\n"
            inputfile += f"%pal nprocs {self.nproc} end\n\n"
            inputfile += f"%maxcore {self.mem}\n\n
            inputfile += f"*xyz &{self.charge} &{self.multiplicity}\n"
            inputfile += self.coords
            inputfile += "*"
            inputfiles.append(("conf_search.inp", inputfile))      

        # optimization input file

        if self.opt:
            inputfile = f"! &{self.optTheory} TightSCF Opt\n\n"
            inputfile += f"%pal nprocs {self.nproc} end\n\n"
            inputfile += f"%maxcore {self.mem}\n\n"
            if self.geomConstraints:
                inputfile += "%geom\n  Constraints\n"
                for constraint in self.optConstraints:
                    inputfile += f" {constraint}\n"
                inputfile += "end\nend\n"

            if self.optExtraBasis:
                inputfile += "%basis\n"
                for element, basis in self.optExtraBasis.items():
                    inputfile += f' NewGTO {element} "{basis}"\n'
                inputfile += "end\n\n"

            if self.confSearch:
                inputfile += f"*xyzfile &{self.charge} &{self.multiplicity} conf_search.globalminimum.xyz"
            else:
                inputfile += f"*xyz &{self.charge} &{self.multiplicity}\n"
                inputfile += self.coords
                inputfile += "*"
            inputfiles.append(("optimization.inp", inputfile))

        # property calculation input file(s)